# Charging Demand per Station — Spain Interurban Network 2027
**IE Sustainability Datathon March 2026 — Iberdrola**

This notebook calculates how many 150 kW chargers each proposed interurban station needs,
based on the province-level EV fleet from notebook 2.2.

**Inputs:**  `outputs/province_demand_2027.csv` (notebook 2.2)

**Outputs:** `outputs/File_2_proposed_stations.csv` — partial File 2 (`grid_status` = PENDING until task 1.3 grid data is integrated)

## 0. Install Dependencies

In [ ]:
# Run only if missing
# !pip install -q pandas numpy matplotlib plotly

## 1. Imports & Configuration

In [1]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
OUT_DIR   = os.path.join(BASE_DIR, 'outputs')
INPUT_CSV = os.path.join(BASE_DIR, 'outputs', 'province_demand_2027.csv')
os.makedirs(OUT_DIR, exist_ok=True)

CHARGER_KW = 150  # Fixed by datathon rules — do not change

print('Paths OK')
print(f'  Input : {INPUT_CSV}')
print(f'  Output: {OUT_DIR}')

Paths OK
  Input : c:\Users\steve\OneDrive - IE University\Term 2\DATATHON_2\Iberdrola\SpanishElecticGrid\notebooks\outputs\province_demand_2027.csv
  Output: c:\Users\steve\OneDrive - IE University\Term 2\DATATHON_2\Iberdrola\SpanishElecticGrid\notebooks\outputs


## 2. Demand Model Assumptions

All parameters below are documented for the Analytical Report.

| Parameter | Value | Source / Justification |
|---|---|---|
| Interurban trip rate | 3% / day | MITMA 2023 Encuesta de Movilidad: avg 0.8 trips/person/day, ~15% interurban; EV owners drive ~20% more |
| Charging need rate | 45% | Avg interurban trip ~180 km one-way (MITMA); round trip 360 km approaches BEV range (350–450 km) |
| Peak-hour share | 15% | DGT hourly traffic distribution: peaks at 09:00 and 17:00–18:00 |
| Session duration | 20 min | 150 kW × 0.33 h = 50 kWh ≈ 200 km extra range (250 Wh/km avg) — Ionity/ACEA 2023 |
| Utilisation target | 75% | Industry standard for profitable highway charging — McKinsey EV Infrastructure 2023 |
| Min chargers/station | 2 | Operational redundancy minimum |
| Max chargers/station | 20 | Practical grid connection ceiling |


In [2]:
# ── Demand model parameters ──────────────────────────────────────────────
INTERURBAN_TRIP_RATE = 0.03
CHARGING_NEED_RATE   = 0.45
PEAK_HOUR_SHARE      = 0.15
SESSION_DURATION_H   = 20 / 60   # 20 minutes
UTILIZATION_TARGET   = 0.75
MIN_CHARGERS         = 2
MAX_CHARGERS         = 20

print('Demand model parameters:')
print(f'  Interurban trip rate  : {INTERURBAN_TRIP_RATE:.0%} of fleet per day')
print(f'  Charging need rate    : {CHARGING_NEED_RATE:.0%} of interurban trips')
print(f'  Peak-hour share       : {PEAK_HOUR_SHARE:.0%} of daily demand')
print(f'  Session duration      : {SESSION_DURATION_H*60:.0f} min at {CHARGER_KW} kW')
print(f'  Utilisation target    : {UTILIZATION_TARGET:.0%}')
print(f'  Charger range         : {MIN_CHARGERS}–{MAX_CHARGERS} per station')

Demand model parameters:
  Interurban trip rate  : 3% of fleet per day
  Charging need rate    : 45% of interurban trips
  Peak-hour share       : 15% of daily demand
  Session duration      : 20 min at 150 kW
  Utilisation target    : 75%
  Charger range         : 2–20 per station


## 3. Load Province Demand (from Notebook 2.2)

Islands (GC, TF, IB) are excluded — they have no interurban road corridors to the mainland.

In [3]:
demand = pd.read_csv(INPUT_CSV).set_index('province_code')

# Exclude islands — no interurban highway connection to mainland
ISLAND_CODES = {'GC', 'TF', 'IB'}
mainland = demand[~demand.index.isin(ISLAND_CODES)].copy()

mainland_total = mainland['ev_fleet_2027'].sum()
island_total   = demand.loc[list(ISLAND_CODES), 'ev_fleet_2027'].sum()

print(f'Provinces loaded : {len(demand)} total | {len(mainland)} mainland | {len(ISLAND_CODES)} islands excluded')
print(f'Mainland EV fleet 2027 : {mainland_total:>10,.0f}')
print(f'Island EV fleet 2027   : {island_total:>10,.0f}  (separate island strategy required)')
print(f'\nTop 10 mainland provinces by EV fleet:')
print(mainland[['province_name', 'auto_community', 'ev_fleet_2027', 'share_pct']]
      .head(10).to_string())

Provinces loaded : 52 total | 49 mainland | 3 islands excluded
Mainland EV fleet 2027 :  1,320,037
Island EV fleet 2027   :     92,603  (separate island strategy required)

Top 10 mainland provinces by EV fleet:
                   province_name        auto_community  ev_fleet_2027  share_pct
province_code                                                                   
M                         Madrid   Comunidad de Madrid         635635    44.9963
B                      Barcelona              Cataluña         184440    13.0564
V              Valencia/València  Comunitat Valenciana          56301     3.9855
A               Alicante/Alacant  Comunitat Valenciana          45556     3.2249
MA                        Málaga             Andalucía          31745     2.2472
BI                       Bizkaia            País Vasco          22951     1.6247
SE                       Sevilla             Andalucía          22554     1.5966
TO                        Toledo    Castilla-La Mancha     

## 4. Interurban Corridor Definitions

We define the primary interurban corridors as classified by the Ministry of Transport.
Each corridor lists:
- The **provinces** it passes through (used to calculate catchment EV demand)
- The **proposed station locations** (lat, lon, descriptive label)

Station spacing follows the **150 km rule**: maximum gap between stations so that a BEV
with 300 km real-world range (conservative estimate) can always reach the next charger
with a 50% battery buffer remaining.

> **Note:** Coordinates are placed at high-traffic mid-corridor points (service area zones,
> major junctions). They will be refined against the Ministry of Transport road dataset
> (task 1.1) before final submission.

In [4]:
# Each entry: (latitude, longitude, descriptive_label)
CORRIDORS = {
    'A-1': {
        'name': 'Autovía del Norte',
        'description': 'Madrid — Burgos — Vitoria — Irún',
        'provinces': ['M', 'BU', 'VI', 'SS'],
        'stations': [
            (41.670, -3.690, 'Aranda de Duero'),
            (42.340, -3.697, 'Burgos Norte'),
            (42.690, -2.940, 'Miranda de Ebro'),
        ]
    },
    'A-2': {
        'name': 'Autovía del Nordeste',
        'description': 'Madrid — Zaragoza — Lleida — Barcelona',
        'provinces': ['M', 'GU', 'Z', 'L', 'B'],
        'stations': [
            (40.950, -2.880, 'Medinaceli'),
            (41.350, -1.640, 'Calatayud'),
            (41.649, -0.887, 'Zaragoza'),
            (41.618,  0.620, 'Lleida'),
        ]
    },
    'A-3': {
        'name': 'Autovía del Este',
        'description': 'Madrid — Cuenca — Valencia',
        'provinces': ['M', 'CU', 'V'],
        'stations': [
            (40.010, -3.010, 'Tarancón'),
            (39.570, -1.890, 'Motilla del Palancar'),
        ]
    },
    'A-4': {
        'name': 'Autovía del Sur',
        'description': 'Madrid — Córdoba — Sevilla — Cádiz',
        'provinces': ['M', 'CR', 'CO', 'SE', 'CA'],
        'stations': [
            (38.990, -3.370, 'Manzanares'),
            (37.890, -4.780, 'Córdoba'),
            (37.383, -5.987, 'Sevilla Norte'),
            (36.690, -6.140, 'Jerez de la Frontera'),
        ]
    },
    'A-5': {
        'name': 'Autovía del Suroeste',
        'description': 'Madrid — Talavera — Mérida — Badajoz',
        'provinces': ['M', 'TO', 'CC', 'BA'],
        'stations': [
            (39.960, -4.830, 'Talavera de la Reina'),
            (39.460, -5.880, 'Trujillo'),
            (38.920, -6.340, 'Mérida'),
        ]
    },
    'A-6': {
        'name': 'Autovía del Noroeste',
        'description': 'Madrid — Ávila — Salamanca — Lugo — A Coruña',
        'provinces': ['M', 'AV', 'SA', 'ZA', 'LU', 'C'],
        'stations': [
            (40.660, -4.700, 'Ávila'),
            (40.966, -5.664, 'Salamanca'),
            (42.000, -5.680, 'Benavente'),
            (43.012, -7.556, 'Lugo'),
        ]
    },
    'AP-7': {
        'name': 'Autopista del Mediterráneo',
        'description': 'Barcelona — Valencia — Alicante — Murcia — Almería — Málaga — Algeciras',
        'provinces': ['B', 'T', 'CS', 'V', 'A', 'MU', 'AL', 'GR', 'MA', 'CA'],
        'stations': [
            (41.119,  1.244, 'Tarragona'),
            (40.470,  0.470, 'Vinaròs'),
            (39.680, -0.270, 'Sagunto'),
            (38.970, -0.180, 'Gandía'),
            (38.345, -0.483, 'Alicante'),
            (37.987, -1.130, 'Murcia'),
            (37.250, -1.860, 'Vera'),
            (36.838, -2.464, 'Almería'),
            (36.720, -4.420, 'Málaga'),
        ]
    },
    'A-8': {
        'name': 'Autovía del Cantábrico',
        'description': 'Bilbao — Santander — Oviedo — Lugo — A Coruña',
        'provinces': ['BI', 'S', 'O', 'LU', 'C'],
        'stations': [
            (43.462, -3.810, 'Santander'),
            (43.362, -5.849, 'Oviedo'),
            (43.012, -7.200, 'Lugo Oeste'),
        ]
    },
    'A-66': {
        'name': 'Autovía de la Plata',
        'description': 'Gijón — Salamanca — Mérida — Sevilla',
        'provinces': ['O', 'SA', 'ZA', 'CC', 'BA', 'SE'],
        'stations': [
            (42.000, -5.700, 'Benavente Sur'),
            (39.476, -6.372, 'Cáceres'),
        ]
    },
    'A-23': {
        'name': 'Autovía Mudéjar',
        'description': 'Zaragoza — Huesca — Jaca',
        'provinces': ['Z', 'HU'],
        'stations': [
            (41.900, -0.520, 'Huesca Sur'),
        ]
    },
    'A-45': {
        'name': 'Autovía de Málaga',
        'description': 'Córdoba — Antequera — Málaga',
        'provinces': ['CO', 'MA'],
        'stations': [
            (37.020, -4.550, 'Antequera'),
        ]
    },
    'A-92': {
        'name': 'Autovía de Andalucía',
        'description': 'Sevilla — Granada — Almería',
        'provinces': ['SE', 'GR', 'AL'],
        'stations': [
            (37.180, -4.010, 'Salinas (Granada)'),
        ]
    },
}

total_stations = sum(len(c['stations']) for c in CORRIDORS.values())
print(f'Corridors defined : {len(CORRIDORS)}')
print(f'Total stations    : {total_stations}')
print()
for road, info in CORRIDORS.items():
    print(f'  {road:6s}  {len(info["stations"])} stations  {info["description"]}')

Corridors defined : 12
Total stations    : 37

  A-1     3 stations  Madrid — Burgos — Vitoria — Irún
  A-2     4 stations  Madrid — Zaragoza — Lleida — Barcelona
  A-3     2 stations  Madrid — Cuenca — Valencia
  A-4     4 stations  Madrid — Córdoba — Sevilla — Cádiz
  A-5     3 stations  Madrid — Talavera — Mérida — Badajoz
  A-6     4 stations  Madrid — Ávila — Salamanca — Lugo — A Coruña
  AP-7    9 stations  Barcelona — Valencia — Alicante — Murcia — Almería — Málaga — Algeciras
  A-8     3 stations  Bilbao — Santander — Oviedo — Lugo — A Coruña
  A-66    2 stations  Gijón — Salamanca — Mérida — Sevilla
  A-23    1 stations  Zaragoza — Huesca — Jaca
  A-45    1 stations  Córdoba — Antequera — Málaga
  A-92    1 stations  Sevilla — Granada — Almería


## 5. Catchment Demand per Station

**Method:** For each corridor, sum the EV fleet of all provinces it passes through.
Divide equally across the stations on that corridor.

This is a conservative proxy — in practice, stations near Madrid will attract more demand
than stations near the terminal end. This assumption is documented and will be refined
with Ministry of Transport traffic count data (task 1.1) if available.

In [5]:
rows = []
for road, info in CORRIDORS.items():
    prov_in_corridor = [p for p in info['provinces'] if p in mainland.index]
    catchment_fleet  = mainland.loc[prov_in_corridor, 'ev_fleet_2027'].sum()
    n_stations       = len(info['stations'])
    fleet_per_stn    = catchment_fleet / n_stations if n_stations > 0 else 0

    for lat, lon, label in info['stations']:
        rows.append({
            'route_segment':    road,
            'road_name':        info['name'],
            'station_label':    label,
            'latitude':         round(lat, 5),
            'longitude':        round(lon, 5),
            'catchment_fleet':  int(catchment_fleet),
            'n_stations_route': n_stations,
            'fleet_per_station': round(fleet_per_stn, 0),
        })

stations = pd.DataFrame(rows)

print(f'Total proposed stations : {len(stations)}')
print()
summary = (
    stations.groupby('route_segment')
    .agg(n_stations=('station_label','count'),
         catchment_fleet=('catchment_fleet','first'),
         fleet_per_station=('fleet_per_station','first'))
    .sort_values('catchment_fleet', ascending=False)
)
summary['catchment_fleet']   = summary['catchment_fleet'].map('{:,.0f}'.format)
summary['fleet_per_station'] = summary['fleet_per_station'].map('{:,.0f}'.format)
print('Catchment demand by corridor:')
print(summary.to_string())

Total proposed stations : 37

Catchment demand by corridor:
               n_stations catchment_fleet fleet_per_station
route_segment                                              
A-2                     4         851,804           212,951
A-3                     2         694,007           347,004
A-4                     4         684,440           171,110
A-5                     3         666,727           222,242
A-6                     4         662,682           165,670
A-1                     3         661,244           220,415
AP-7                    9         401,398            44,600
A-8                     3          65,367            21,789
A-66                    2          51,205            25,602
A-92                    1          41,454            41,454
A-45                    1          39,144            39,144
A-23                    1          20,341            20,341


## 6. Charger Sizing

For each station, we calculate the number of 150 kW chargers needed using the demand model:

```
daily_sessions  = fleet_per_station × trip_rate × charging_need_rate
peak_sessions   = daily_sessions × peak_hour_share
n_chargers      = ceil( peak_sessions × session_duration_h / utilisation_target )
n_chargers      = clamp(n_chargers, MIN=2, MAX=20)
demand_kw       = n_chargers × 150
```

In [6]:
def size_station(fleet_per_station: float) -> int:
    """Return number of 150 kW chargers needed at a station."""
    daily_sessions = fleet_per_station * INTERURBAN_TRIP_RATE * CHARGING_NEED_RATE
    peak_sessions  = daily_sessions * PEAK_HOUR_SHARE
    raw            = math.ceil(peak_sessions * SESSION_DURATION_H / UTILIZATION_TARGET)
    return max(MIN_CHARGERS, min(MAX_CHARGERS, raw))

stations['n_chargers_proposed'] = stations['fleet_per_station'].apply(size_station)
stations['estimated_demand_kw'] = stations['n_chargers_proposed'] * CHARGER_KW

# grid_status is PENDING until colleague integrates grid capacity data (task 1.3)
stations['grid_status'] = 'PENDING'

print('Charger sizing complete.')
print(f'\nChargers per station — distribution:')
print(stations['n_chargers_proposed'].value_counts().sort_index().to_string())
print(f'\nTotal installed capacity : {stations["estimated_demand_kw"].sum():,.0f} kW')
print(f'Average per station      : {stations["estimated_demand_kw"].mean():,.0f} kW')
print(f'\nTop 10 highest-demand stations:')
print(stations.nlargest(10, 'estimated_demand_kw')[
    ['route_segment', 'station_label', 'fleet_per_station', 'n_chargers_proposed', 'estimated_demand_kw']
].to_string(index=False))

Charger sizing complete.

Chargers per station — distribution:
n_chargers_proposed
19     1
20    36

Total installed capacity : 110,850 kW
Average per station      : 2,996 kW

Top 10 highest-demand stations:
route_segment        station_label  fleet_per_station  n_chargers_proposed  estimated_demand_kw
          A-1      Aranda de Duero           220415.0                   20                 3000
          A-1         Burgos Norte           220415.0                   20                 3000
          A-1      Miranda de Ebro           220415.0                   20                 3000
          A-2           Medinaceli           212951.0                   20                 3000
          A-2            Calatayud           212951.0                   20                 3000
          A-2             Zaragoza           212951.0                   20                 3000
          A-2               Lleida           212951.0                   20                 3000
          A-3          

## 7. Build File 2 — Proposed Charging Locations

This is the core output required by the datathon (Deliverable 2, File 2).

| Column | Status |
|---|---|
| `location_id` | Done |
| `latitude` | Done |
| `longitude` | Done |
| `route_segment` | Done |
| `n_chargers_proposed` | Done |
| `grid_status` | **PENDING** — requires grid capacity data from task 1.3 |

> Once the grid capacity CSVs (i-DE, Endesa, Viesgo) are available, run the
> **Grid Status Integration** cell at the bottom of this notebook to replace PENDING
> with Sufficient / Moderate / Congested.

In [7]:
stations['location_id'] = ['IBE_{:03d}'.format(i+1) for i in range(len(stations))]

FILE2_COLS = ['location_id', 'latitude', 'longitude', 'route_segment',
              'n_chargers_proposed', 'estimated_demand_kw', 'grid_status']

file2 = stations[FILE2_COLS].copy()

print('File 2 — structure verification:')
print(file2.dtypes)
print(f'\nRows : {len(file2)}')
print(f'\nFirst 15 rows:')
print(file2.head(15).to_string(index=False))

File 2 — structure verification:
location_id             object
latitude               float64
longitude              float64
route_segment           object
n_chargers_proposed      int64
estimated_demand_kw      int64
grid_status             object
dtype: object

Rows : 37

First 15 rows:
location_id  latitude  longitude route_segment  n_chargers_proposed  estimated_demand_kw grid_status
    IBE_001    41.670     -3.690           A-1                   20                 3000     PENDING
    IBE_002    42.340     -3.697           A-1                   20                 3000     PENDING
    IBE_003    42.690     -2.940           A-1                   20                 3000     PENDING
    IBE_004    40.950     -2.880           A-2                   20                 3000     PENDING
    IBE_005    41.350     -1.640           A-2                   20                 3000     PENDING
    IBE_006    41.649     -0.887           A-2                   20                 3000     PENDING
  

## 8. Interactive Map — Proposed Stations

Stations are shown in **grey** while `grid_status` is pending.
After task 1.3 integration they will turn **green** (Sufficient), **yellow** (Moderate) or **red** (Congested).

Marker **size** encodes the number of chargers proposed at that location.

In [8]:
COLOR_MAP = {
    'Sufficient': '#27ae60',
    'Moderate':   '#f39c12',
    'Congested':  '#e74c3c',
    'PENDING':    '#95a5a6',
}

fig = px.scatter_mapbox(
    file2,
    lat='latitude',
    lon='longitude',
    color='grid_status',
    color_discrete_map=COLOR_MAP,
    size='n_chargers_proposed',
    size_max=22,
    hover_name='location_id',
    hover_data={
        'route_segment':       True,
        'n_chargers_proposed': True,
        'estimated_demand_kw': ':,.0f',
        'grid_status':         True,
        'latitude':            False,
        'longitude':           False,
    },
    mapbox_style='carto-positron',
    zoom=4.5,
    center={'lat': 40.2, 'lon': -3.5},
    title='Proposed Charging Stations — Spain 2027  (grid status: PENDING)',
    labels={
        'n_chargers_proposed': 'Chargers',
        'estimated_demand_kw': 'Demand (kW)',
        'grid_status':         'Grid Status',
        'route_segment':       'Road',
    },
    width=1050,
    height=660,
)
fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

## 9. Output — Save File 2

The CSV is saved to `outputs/File_2_proposed_stations.csv`.

After task 1.3, re-run the **Grid Status Integration** cell (Section 10) and re-save.

In [ ]:
out_path = os.path.join(OUT_DIR, 'File_2_proposed_stations.csv')
file2.to_csv(out_path, index=False, encoding='utf-8')

print(f'Saved : {out_path}')
print(f'Rows  : {len(file2)}')
print()
print('Column compliance check (datathon required fields):')
required = ['location_id', 'latitude', 'longitude',
            'route_segment', 'n_chargers_proposed', 'grid_status']
for col in required:
    status = 'OK     ' if col in file2.columns else 'MISSING'
    val    = file2[col].iloc[0] if col in file2.columns else '---'
    print(f'  {status}  {col:<25s}  example: {val}')
print()
print('Full File 2:')
print(file2.to_string(index=False))

## 10. Grid Status Integration  *(run after task 1.3)*

Once your colleague provides the grid capacity CSVs (i-DE, Endesa, Viesgo), run this cell.

It will:
1. Load each distributor's substation locations and available capacity (MW)
2. For each proposed station, find the **nearest substation** (haversine distance)
3. Compare `estimated_demand_kw` against available capacity using the thresholds below
4. Assign `grid_status` and re-save File 2
5. Auto-generate **File 3** (only Moderate + Congested rows)

**Threshold justification** (document in Analytical Report):

| grid_status | Condition | Rationale |
|---|---|---|
| Sufficient | demand_kw < 50% of available capacity (MW × 1000) | Substation absorbs load with headroom |
| Moderate | 50–80% of available capacity | Load is significant; reinforcement advisable |
| Congested | > 80% of available capacity | Grid reinforcement mandatory before deployment |

In [ ]:
# ── PLACEHOLDER — run after task 1.3 grid data is available ──────────────
#
# GRID_IDE_PATH    = 'path/to/ide_capacity.csv'
# GRID_ENDESA_PATH = 'path/to/endesa_capacity.csv'
# GRID_VIESGO_PATH = 'path/to/viesgo_capacity.csv'
#
# Thresholds (adjust after reviewing distributor data ranges)
THRESHOLD_MODERATE  = 0.50   # demand > 50% of substation capacity
THRESHOLD_CONGESTED = 0.80   # demand > 80% of substation capacity

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

def assign_grid_status(station_row, substations_df,
                        lat_col='lat', lon_col='lon', cap_col='available_mw',
                        distributor_col='distributor'):
    """
    Find nearest substation and classify grid_status.
    substations_df must have columns: lat_col, lon_col, cap_col, distributor_col
    """
    dists = substations_df.apply(
        lambda r: haversine_km(station_row['latitude'], station_row['longitude'],
                               r[lat_col], r[lon_col]), axis=1)
    nearest = substations_df.loc[dists.idxmin()]
    ratio   = (station_row['estimated_demand_kw'] / 1000) / nearest[cap_col]  # kW → MW

    if ratio > THRESHOLD_CONGESTED:
        status = 'Congested'
    elif ratio > THRESHOLD_MODERATE:
        status = 'Moderate'
    else:
        status = 'Sufficient'

    return pd.Series({'grid_status': status,
                      'distributor_network': nearest[distributor_col],
                      'nearest_substation_km': round(dists.min(), 2)})

print('Grid status integration functions ready.')
print('Waiting for grid capacity CSVs from task 1.3.')
print()
print('When data is available, replace the paths above and run:')
print('  substations = pd.concat([ide, endesa, viesgo], ignore_index=True)')
print('  grid_cols   = file2.apply(assign_grid_status, substations_df=substations, axis=1)')
print('  file2[["grid_status","distributor_network"]] = grid_cols[["grid_status","distributor_network"]]')
print('  # Then re-save File 2 and generate File 3 below')

## 11. File 3 — Friction Points  *(auto-generated after Section 10)*

File 3 contains only rows from File 2 where `grid_status` is **Moderate** or **Congested**.
It adds two extra columns: `distributor_network` and `bottleneck_id`.

In [ ]:
# ── Run after Section 10 assigns real grid_status values ─────────────────

friction = file2[file2['grid_status'].isin(['Moderate', 'Congested'])].copy()

if len(friction) == 0:
    print('No friction points yet — grid_status is still PENDING.')
    print('Re-run after Section 10 grid integration is complete.')
else:
    friction['bottleneck_id'] = ['FRIC_{:03d}'.format(i+1) for i in range(len(friction))]

    FILE3_COLS = ['bottleneck_id', 'latitude', 'longitude', 'route_segment',
                  'distributor_network', 'estimated_demand_kw', 'grid_status']

    # distributor_network populated by assign_grid_status() in Section 10
    file3 = friction[FILE3_COLS].copy()

    file3_path = os.path.join(OUT_DIR, 'File_3_friction_points.csv')
    file3.to_csv(file3_path, index=False, encoding='utf-8')

    print(f'Saved : {file3_path}')
    print(f'Rows  : {len(file3)}  ({len(friction[friction["grid_status"]=="Congested"])} Congested,'
          f' {len(friction[friction["grid_status"]=="Moderate"])} Moderate)')
    print(file3.to_string(index=False))

## 12. Summary — Notebook 2.3 Status

| Item | Status |
|---|---|
| Province demand loaded | Done |
| Demand parameters defined & justified | Done |
| Highway corridors defined (12 routes) | Done |
| Catchment demand per station | Done |
| `n_chargers_proposed` calculated | Done |
| `estimated_demand_kw` = n_chargers × 150 kW | Done |
| File 2 saved (partial) | Done |
| `grid_status` assignment | **Pending task 1.3** |
| File 3 (friction points) | **Pending task 1.3** |
| File 1 KPI scorecard | **Pending task 1.2 + 1.3** |